# Classical and optimal designs

Central-composite, Box–Behnken, factorial and fractional-factorial, Latin hypercube — plus
D/A/E-optimal point exchange. On a nonlinear surface optimality is local, so
`bayesian_criterion` averages the criterion over prior draws (review B11); a single draw is the
local special case.

In [ ]:
import numpy as np

from axiom.core import D, Outcome, Treatment
from axiom.surface import (
    Bounds, Criterion, Design, HillKernel, Surface, SurfaceSpec, a_criterion, bayesian_criterion,
    box_behnken, central_composite, d_criterion, defining_relation, e_criterion, equal_spacing,
    fractional_factorial, full_factorial, latin_hypercube, optimal_exchange,
)

In [ ]:
bounds = Bounds(treatments=("a", "b"), low=(0.0, 0.0), high=(100.0, 40.0))
ccd: Design = central_composite(bounds, alpha="rotatable", center_points=3)
print(ccd.kind, ccd.n, ccd.detail)
print(ccd.as_frame().head())

In [ ]:
b3 = Bounds(treatments=("a", "b", "c"), low=(0.0,) * 3, high=(1.0,) * 3)
print("box-behnken k=3:", box_behnken(b3, center_points=2).n)
print("3^2 factorial:", full_factorial(bounds, 3).n)
ff = fractional_factorial(Bounds(treatments=("A", "B", "C", "D"), low=(0.0,) * 4, high=(1.0,) * 4), generators=("D=ABC",))
print("2^(4-1):", ff.n, ff.detail, defining_relation(("D=ABC",)))
lhs = latin_hypercube(bounds, 12, seed=0)
print("LHS:", lhs.n, "| equal spacing:", equal_spacing(bounds, 4).n)

## Criteria and optimal exchange

Criteria act on the design matrix of the linearized surface. `optimal_exchange` runs a
Fedorov-style point exchange among candidate points; it returns `Unsupported` rather than a
number when no non-singular design is reachable.

In [ ]:
spec = SurfaceSpec(
    name="two_hill",
    treatments=(Treatment(name="a", dimension=D.currency, unit="USD"), Treatment(name="b", dimension=D.currency, unit="USD")),
    outcome=Outcome(name="y", dimension=D.outcome),
    kernels={"a": HillKernel(reference_dose=50.0), "b": HillKernel(reference_dose=20.0)},
)
surface = Surface(spec)
theta = {"alpha": 1.0, "k_a": 50.0, "s_a": 2.0, "beta_a": 10.0, "k_b": 20.0, "s_b": 1.5, "beta_b": 5.0, "sigma": 1.0}
X = surface.linearize(ccd.doses(), theta)
crit: Criterion = "d"
print("D:", round(d_criterion(X), 3), "A:", round(a_criterion(X), 3), "E:", round(e_criterion(X), 4))

In [ ]:
rng = np.random.default_rng(0)
prior_draws = [{**theta, "k_a": float(rng.lognormal(np.log(50), 0.3)), "s_a": float(rng.gamma(4, 0.5))} for _ in range(8)]
candidates = full_factorial(bounds, 7)
naive = equal_spacing(bounds, 3)
opt = optimal_exchange(surface, candidates, n=naive.n, theta_draws=prior_draws, criterion=crit, seed=0)
print("naive   :", round(bayesian_criterion(surface, naive, prior_draws), 3))
print("optimal :", round(bayesian_criterion(surface, opt, prior_draws), 3), "| local (one draw):", round(bayesian_criterion(surface, opt, [theta]), 3))
print(opt.detail)